# Debug game similarity and annual feature importance

Set the shared YAML path below and run all cells in Databricks. The first panel recomputes neighbors for one target. The second uses observed commercial results to compare features across a calendar year's brackets. Both read the AI and commercial/Eilers source versions resolved by notebooks 05 and 06. All debug calculations stay in this session.

Use **Find like games** or **Analyze year** after editing the controls. Each output identifies the selection it used.


In [0]:
CONFIG_PATH = "configs/experiments.yaml"


In [0]:
"""Validated YAML configuration and semantic work identities for notebook 05."""
import copy
import hashlib
import json
import math
import re
from datetime import date, datetime
from numbers import Integral, Real
from pathlib import Path

SIMILARITY_IMPLEMENTATION_VERSION = "05_batch_yaml_v1_2026_11_19"
WEIGHT_GROUPS = ("mechanics", "themes", "reels", "persistence", "ways", "months_since_release")
PARENT_PROCESSES = {"p1_parent_games_only", "p2_current", "p3_include_parent"}


def config_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False,
                      default=lambda x: x.isoformat() if isinstance(x, (date, datetime)) else str(x))


def config_sha256(value):
    return hashlib.sha256(config_json(value).encode("utf-8")).hexdigest()


def quoted_table(name):
    return ".".join("`" + p.replace("`", "``") + "`" for p in name.split("."))


def _keys(value, allowed, where, required=()):
    if not isinstance(value, dict):
        raise ValueError(f"{where} must be a mapping.")
    unknown, missing = set(value) - set(allowed), set(required) - set(value)
    if unknown or missing:
        raise ValueError(f"{where}: unknown keys={sorted(unknown)}, missing keys={sorted(missing)}")


def _text(value, where):
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{where} must be a nonempty string.")
    return value.strip()


def _integer(value, where, minimum=0):
    if isinstance(value, bool) or not isinstance(value, Integral) or value < minimum:
        raise ValueError(f"{where} must be an integer >= {minimum}.")
    return int(value)


def _date(value, where):
    if value is None:
        return None
    if isinstance(value, datetime):
        value = value.date()
    if isinstance(value, date):
        return value.isoformat()
    try:
        return date.fromisoformat(str(value)).isoformat()
    except ValueError as exc:
        raise ValueError(f"{where} must be YYYY-MM-DD or null.") from exc


def _table(value, where):
    value = _text(value, where)
    if len(value.split(".")) != 3 or not all(re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", x) for x in value.split(".")):
        raise ValueError(f"{where} must be a three-part catalog.schema.table name.")
    return value


def validate_similarity_settings(settings):
    allowed = {"raw_weights", "distance_method", "parent_game_process", "neighbor_limits",
               "neighbor_released_after", "minimum_age_gap_months"}
    _keys(settings, allowed, "similarity settings", allowed)
    settings = copy.deepcopy(settings)
    settings["distance_method"] = _text(settings["distance_method"], "distance_method").lower()
    if settings["distance_method"] not in {"cosine", "euclidean"}:
        raise ValueError("distance_method must be cosine or euclidean.")
    if settings["parent_game_process"] not in PARENT_PROCESSES:
        raise ValueError(f"Unknown parent_game_process: {settings['parent_game_process']!r}")
    _keys(settings["raw_weights"], WEIGHT_GROUPS, "raw_weights", WEIGHT_GROUPS)
    for key, value in settings["raw_weights"].items():
        if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(float(value)) or value < 0:
            raise ValueError(f"raw_weights.{key} must be a finite nonnegative number.")
        settings["raw_weights"][key] = float(value)
    if not any(settings["raw_weights"].values()):
        raise ValueError("At least one feature-group weight must be positive.")
    _keys(settings["neighbor_limits"], {"sales", "rental", "eilers"}, "neighbor_limits", {"sales", "rental", "eilers"})
    settings["neighbor_limits"] = {k: _integer(v, f"neighbor_limits.{k}") for k, v in settings["neighbor_limits"].items()}
    settings["minimum_age_gap_months"] = _integer(settings["minimum_age_gap_months"], "minimum_age_gap_months")
    settings["neighbor_released_after"] = _date(settings["neighbor_released_after"], "neighbor_released_after")
    if settings["neighbor_released_after"] is None:
        raise ValueError("neighbor_released_after must be a date.")
    return settings


def load_experiment_config(config_path, require_enabled=True):
    try:
        import yaml
    except ImportError as exc:
        raise ImportError("PyYAML is required. Install PyYAML on your Databricks environment, then rerun.") from exc

    # Reject duplicate YAML keys rather than silently replacing a weight/setting.
    class UniqueLoader(yaml.SafeLoader):
        pass
    def unique_mapping(loader, node, deep=False):
        result = {}
        for key_node, value_node in node.value:
            key = loader.construct_object(key_node, deep=deep)
            if key in result:
                raise ValueError(f"Duplicate YAML key {key!r} on line {key_node.start_mark.line + 1}.")
            result[key] = loader.construct_object(value_node, deep=deep)
        return result
    UniqueLoader.add_constructor(yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG, unique_mapping)
    path = Path(config_path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f"Configuration not found: {path}. Put experiments.yaml in configs beside this notebook, or set CONFIG_PATH to its absolute /Workspace/... path.")
    original_text = path.read_text(encoding="utf-8")
    raw = yaml.load(original_text, Loader=UniqueLoader)
    sections = {"schema_version", "data", "outputs", "tracking", "monitoring", "similarity_defaults", "experiments"}
    # Downstream stages share this file without changing 05's semantic inputs.
    _keys(raw, sections | {"forecast", "evaluation"}, "configuration", sections)
    if isinstance(raw["schema_version"], bool) or raw["schema_version"] != 1:
        raise ValueError("Only schema_version: 1 is supported.")
    _keys(raw["data"], {"revision", "ai_attributes_table", "source_version", "prediction_as_of_date"}, "data",
          {"revision", "ai_attributes_table", "source_version", "prediction_as_of_date"})
    data = copy.deepcopy(raw["data"])
    data["revision"] = _text(data["revision"], "data.revision")
    data["ai_attributes_table"] = _table(data["ai_attributes_table"], "data.ai_attributes_table")
    if data["source_version"] is not None:
        data["source_version"] = _integer(data["source_version"], "data.source_version")
    data["prediction_as_of_date"] = _date(data["prediction_as_of_date"], "data.prediction_as_of_date")
    output_keys = {"like_games_table", "runs_table", "attempts_table"}
    _keys(raw["outputs"], output_keys, "outputs", output_keys)
    outputs = {k: _table(v, f"outputs.{k}") for k, v in raw["outputs"].items()}
    if len(set(outputs.values())) != len(outputs) or data["ai_attributes_table"] in outputs.values():
        raise ValueError("Output tables must be distinct and cannot be the AI source table.")
    _keys(raw["tracking"], {"experiment_path"}, "tracking", {"experiment_path"})
    tracking_path = _text(raw["tracking"]["experiment_path"], "tracking.experiment_path")
    if not tracking_path.startswith("/"):
        raise ValueError("tracking.experiment_path must start with /.")
    _keys(raw["monitoring"], {"history_rows"}, "monitoring", {"history_rows"})
    history_rows = _integer(raw["monitoring"]["history_rows"], "monitoring.history_rows", 1)
    defaults = validate_similarity_settings(raw["similarity_defaults"])
    if not isinstance(raw["experiments"], list) or not raw["experiments"]:
        raise ValueError("experiments must be a nonempty list.")
    experiments, names, identities = [], set(), {}
    for index, item in enumerate(raw["experiments"]):
        _keys(item, {"name", "enabled", "overrides"}, f"experiments[{index}]", {"name"})
        name = _text(item["name"], f"experiments[{index}].name")
        if name in names:
            raise ValueError(f"Duplicate experiment name: {name!r}")
        names.add(name)
        enabled = item.get("enabled", True)
        if not isinstance(enabled, bool):
            raise ValueError(f"{name}.enabled must be true or false.")
        overrides = item.get("overrides", {})
        _keys(overrides, defaults, f"{name}.overrides")
        merged = copy.deepcopy(defaults)
        for key, value in overrides.items():
            if key in {"raw_weights", "neighbor_limits"}:
                _keys(value, defaults[key], f"{name}.overrides.{key}")
                merged[key].update(copy.deepcopy(value))
            else:
                merged[key] = copy.deepcopy(value)
        merged = validate_similarity_settings(merged)
        identity = config_sha256(merged)
        if enabled and identity in identities:
            raise ValueError(f"Enabled experiments {identities[identity]!r} and {name!r} have identical settings. Keep one enabled; names do not create different experiments.")
        if enabled:
            identities[identity] = name
        experiments.append({"name": name, "enabled": enabled, "settings": merged})
    if require_enabled and not any(x["enabled"] for x in experiments):
        raise ValueError("Enable at least one experiment in the YAML file.")
    return {"data": data, "outputs": outputs, "tracking": {"experiment_path": tracking_path},
            "monitoring": {"history_rows": history_rows}, "experiments": experiments,
            "config_path": str(path), "config_text": original_text,
            "yaml_sha256": hashlib.sha256(original_text.encode("utf-8")).hexdigest()}


def resolve_similarity_source_version(spark, config):
    """A data revision pins one source snapshot across later manually run batches."""
    from pyspark.sql import functions as F
    data, runs = config["data"], config["outputs"]["runs_table"]
    pinned = []
    if spark.catalog.tableExists(runs):
        ledger = spark.table(runs)
        if "metadata_json" not in ledger.columns:
            raise ValueError("Incompatible similarity status table; perform the one-time reset first.")
        pinned = [int(r["source_table_version"]) for r in ledger.where(
            (F.get_json_object("metadata_json", "$.data_revision") == data["revision"])
            & (F.col("source_path") == data["ai_attributes_table"])
        ).select("source_table_version").distinct().collect()]
    if len(set(pinned)) > 1:
        raise ValueError("This data revision has multiple source snapshots. Choose a new revision after resolving the history.")
    if pinned:
        version = pinned[0]
        if data["source_version"] is not None and data["source_version"] != version:
            raise ValueError(f"Revision {data['revision']!r} is already pinned to Delta version {version}. Change data.revision to use another version.")
        return version
    if data["source_version"] is not None:
        return data["source_version"]
    return int(spark.sql(f"DESCRIBE HISTORY {quoted_table(data['ai_attributes_table'])}").agg(F.max("version").alias("version")).first()["version"])

FORECAST_SOURCE_KEYS = {"sales", "lease", "performance"}
FORECAST_OUTPUT_KEYS = {"demand", "performance", "neighbors", "runs", "attempts"}
FORECAST_SETTING_KEYS = {"distance_cutoff", "minimum_neighbors", "horizon_months", "slot_counts",
                         "lease_value_column", "lease_metric_name", "lease_curve_label", "performance_metrics"}


def load_forecast_config(config_path):
    # Shared loader rejects duplicate YAML keys and validates the existing 05
    # sections. 05 ignores this downstream section for work identity purposes.
    common = load_experiment_config(config_path, require_enabled=False)
    import yaml
    raw = yaml.safe_load(common["config_text"])
    forecast = raw.get("forecast")
    forecast_keys = {"data_revision", "sources", "source_versions", "settings", "outputs", "tracking", "selection", "monitoring"}
    _keys(forecast, forecast_keys, "forecast", forecast_keys)
    revision = _text(forecast["data_revision"], "forecast.data_revision")
    _keys(forecast["sources"], FORECAST_SOURCE_KEYS, "forecast.sources", FORECAST_SOURCE_KEYS)
    sources = {k: _table(v, f"forecast.sources.{k}") for k, v in forecast["sources"].items()}
    _keys(forecast["source_versions"], FORECAST_SOURCE_KEYS, "forecast.source_versions", FORECAST_SOURCE_KEYS)
    versions = {k: None if v is None else _integer(v, f"forecast.source_versions.{k}")
                for k, v in forecast["source_versions"].items()}
    _keys(forecast["outputs"], FORECAST_OUTPUT_KEYS, "forecast.outputs", FORECAST_OUTPUT_KEYS)
    outputs = {k: _table(v, f"forecast.outputs.{k}") for k, v in forecast["outputs"].items()}
    protected_inputs = {common["data"]["ai_attributes_table"], *sources.values(), *common["outputs"].values()}
    if len(set(outputs.values())) != len(outputs) or set(outputs.values()) & protected_inputs:
        raise ValueError("Forecast output tables must be distinct and cannot replace any source or similarity tables.")
    _keys(forecast["settings"], FORECAST_SETTING_KEYS, "forecast.settings", FORECAST_SETTING_KEYS)
    settings = copy.deepcopy(forecast["settings"])
    cutoff = settings["distance_cutoff"]
    if isinstance(cutoff, bool) or not isinstance(cutoff, Real) or not math.isfinite(float(cutoff)) or cutoff < 0:
        raise ValueError("forecast.settings.distance_cutoff must be finite and >= 0.")
    settings["distance_cutoff"] = float(cutoff)
    settings["minimum_neighbors"] = _integer(settings["minimum_neighbors"], "minimum_neighbors")
    settings["horizon_months"] = _integer(settings["horizon_months"], "horizon_months", 1)
    slots = settings["slot_counts"]
    if not isinstance(slots, list) or any(isinstance(x, bool) or not isinstance(x, Real) or not math.isfinite(float(x)) or x <= 0 for x in slots):
        raise ValueError("slot_counts must be a list of finite positive numbers; [] means all positive counts.")
    if len(set(slots)) != len(slots):
        raise ValueError("slot_counts cannot contain duplicates.")
    settings["slot_counts"] = sorted(float(x) for x in slots)
    for key in ["lease_value_column", "lease_metric_name", "lease_curve_label"]:
        settings[key] = _text(settings[key], key)
    if not isinstance(settings["performance_metrics"], list) or not settings["performance_metrics"]:
        raise ValueError("performance_metrics must be a nonempty list.")
    settings["performance_metrics"] = [_text(x, "performance_metrics entry") for x in settings["performance_metrics"]]
    if len(set(settings["performance_metrics"])) != len(settings["performance_metrics"]):
        raise ValueError("performance_metrics cannot contain duplicates.")
    settings["performance_metrics"] = sorted(settings["performance_metrics"])
    _keys(forecast["tracking"], {"experiment_path"}, "forecast.tracking", {"experiment_path"})
    mlflow_path = _text(forecast["tracking"]["experiment_path"], "forecast.tracking.experiment_path")
    if not mlflow_path.startswith("/"):
        raise ValueError("forecast.tracking.experiment_path must be an absolute workspace path.")
    _keys(forecast["selection"], {"similarity_experiment_ids"}, "forecast.selection", {"similarity_experiment_ids"})
    ids = forecast["selection"]["similarity_experiment_ids"]
    if not isinstance(ids, list):
        raise ValueError("similarity_experiment_ids must be a list; [] discovers all completed 05 experiments.")
    ids = [_text(x, "similarity_experiment_ids entry") for x in ids]
    if len(set(ids)) != len(ids):
        raise ValueError("similarity_experiment_ids cannot contain duplicates.")
    _keys(forecast["monitoring"], {"history_rows"}, "forecast.monitoring", {"history_rows"})
    return {"common": common, "data_revision": revision, "sources": sources, "source_versions": versions,
            "settings": settings, "outputs": outputs, "tracking": {"experiment_path": mlflow_path},
            "selection": {"similarity_experiment_ids": ids},
            "monitoring": {"history_rows": _integer(forecast["monitoring"]["history_rows"], "forecast.monitoring.history_rows", 1)}}


def resolve_forecast_source_versions(spark, config):
    from pyspark.sql import functions as F
    table = config["outputs"]["runs"]
    matches = []
    if spark.catalog.tableExists(table):
        ledger = spark.table(table)
        if "metadata_json" not in ledger.columns:
            raise ValueError("Legacy forecast manifest found. Reset scope '06_07' before using new 06; preserve 05.")
        matches = ledger.where(F.get_json_object("metadata_json", "$.data_revision") == config["data_revision"]).select(
            "source_versions_json", "config_json").distinct().collect()
    versions = None
    for row in matches:
        previous_config = json.loads(row["config_json"])
        if previous_config["source_paths"] != config["sources"]:
            raise ValueError("This forecast data revision is already assigned to other source tables. Choose another forecast.data_revision.")
        candidate = {k: int(v) for k, v in json.loads(row["source_versions_json"]).items()}
        if versions is not None and candidate != versions:
            raise ValueError("This forecast revision contains conflicting source snapshots; resolve its history before running.")
        versions = candidate
    if versions is not None:
        for key, requested in config["source_versions"].items():
            if requested is not None and requested != versions[key]:
                raise ValueError(f"Revision is pinned to {key} version {versions[key]}; change forecast.data_revision to refresh.")
        return versions
    return {key: int(requested) if requested is not None else int(
        spark.sql(f"DESCRIBE HISTORY {quoted_table(config['sources'][key])}").agg(F.max("version").alias("v")).first()["v"])
        for key, requested in config["source_versions"].items()}


In [0]:
import numpy as np
import pandas as pd

def normalize_game_reference(value):
    if value is None or pd.isna(value):
        return None
    key = " ".join(str(value).strip().lower().split())
    return None if key in {"", "none", "null", "<na>", "nan", "na", "n/a", "__unknown__"} else key


def as_bool(value):
    if value is None or pd.isna(value):
        return False
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def _numeric_feature_expr(source_columns, root_col, preferred_names):
    from pyspark.sql import functions as F
    from pyspark.sql.types import StructType, NumericType, StringType

    def extract(expression):
        value = F.regexp_extract(expression.cast("string"), r"(-?\d+(?:\.\d+)?)", 1)
        return F.when(F.length(value) > 0, value.cast("double"))

    def leaves(data_type, prefix=()):
        if isinstance(data_type, StructType):
            return [leaf for field in data_type.fields
                    for leaf in leaves(field.dataType, prefix + (field.name,))]
        return [(prefix, data_type)]

    data_type = source_columns[root_col].dataType
    if isinstance(data_type, NumericType):
        return F.col(root_col).cast("double")
    if isinstance(data_type, StringType):
        return extract(F.col(root_col))
    if isinstance(data_type, StructType):
        fields = leaves(data_type)
        chosen = None
        for preferred in preferred_names:
            chosen = next(((path, typ) for path, typ in fields
                           if path[-1].lower() == preferred
                           and isinstance(typ, (NumericType, StringType))), None)
            if chosen is not None:
                break
        if chosen is None:
            chosen = next(((path, typ) for path, typ in fields if isinstance(typ, NumericType)), None)
        if chosen is None:
            chosen = next(((path, typ) for path, typ in fields if isinstance(typ, StringType)), None)
        if chosen is not None:
            path, leaf_type = chosen
            expression = F.col(root_col)
            for part in path:
                expression = expression.getField(part)
            return expression.cast("double") if isinstance(leaf_type, NumericType) else extract(expression)
    raise ValueError(f"Cannot extract a numeric feature from {root_col}: {data_type}")


def prepare_games(source_df):
    """Project numeric features and deduplicate the pinned Spark source once.

    Actual/match availability affects candidate pools, never target retention.
    The returned population is independent of experiment weights and settings.
    """
    from pyspark.sql import functions as F

    columns = {field.name: field for field in source_df.schema.fields}
    missing = {"game_name", "join_game_name_key"} - set(columns)
    if missing:
        raise ValueError(f"AI attributes missing canonical columns: {sorted(missing)}")
    themes = sorted(c for c in columns if c.startswith("theme_") and c.endswith("_score")
                    and c != "theme_presence_score")
    mechanics = sorted(c for c in columns if c.startswith("mechanic_") and c.endswith("_present"))
    mechanics = mechanics or sorted(c for c in columns if c.startswith("mechanic_") and c.endswith("_score"))
    persistence = [c for c in ["persistence_score", "perceived_persistence_score",
                              "persistence", "perceived_persistence"] if c in columns]
    reels = next((c for c in ["number_of_reels", "reels"] if c in columns), None)
    ways = next((c for c in ["ways", "lines_or_ways", "paylines_or_ways_count", "lines"] if c in columns), None)
    if not themes or not mechanics or not persistence or reels is None or ways is None:
        raise ValueError("Expected theme, mechanic, persistence, reels, and ways/lines feature columns.")
    own_column = "own_status" if "own_status" in columns else "owned_status" if "owned_status" in columns else None
    projected = [F.col("game_name"), F.col("join_game_name_key")]
    metadata = ["parent_game", "game_category", "parent_own_status", "game_classification",
                "game_release_date", "max_months_since_release",
                "to_sales_name_match", "to_rental_name_match"]
    for name in metadata:
        projected.append(F.col(name) if name in columns else F.lit(None).cast("string").alias(name))
    projected.append(F.col(own_column).alias("own_status") if own_column else F.lit(None).cast("string").alias("own_status"))
    # Preserve the previous Eilers fallback when a separate mapping column is absent.
    projected.append((F.col("to_eilers_ts_name_match") if "to_eilers_ts_name_match" in columns
                      else F.col("game_name")).alias("to_eilers_ts_name_match"))
    for name in ["has_sales_match", "has_rental_match", "has_eilers_ts_match"]:
        projected.append(F.col(name) if name in columns else F.lit(False).alias(name))
    projected.extend([
        _numeric_feature_expr(columns, reels, ["number_of_reels", "reel_count", "count", "base_reels", "reels", "value", "score"]).alias("reels_clean"),
        _numeric_feature_expr(columns, ways, ["value", "ways", "lines", "count", "paylines_or_ways_count", "score"]).alias("ways_clean"),
    ])
    projected.extend(_numeric_feature_expr(columns, c, ["value", "score", "persistence", "persistence_score"]).alias(c)
                     for c in persistence)
    projected.extend(F.expr("try_cast(`" + c.replace("`", "``") + "` AS DOUBLE)").alias(c)
                     for c in themes + mechanics)
    groups = {"mechanics": mechanics, "themes": themes, "reels": ["reels_clean"],
              "persistence": persistence, "ways": ["ways_clean"],
              "months_since_release": ["max_months_since_release"]}
    return prepare_game_frame(source_df.select(*projected).toPandas(), groups)


def prepare_game_frame(frame, feature_groups):
    """Pure pandas preparation after Spark feature projection; useful for tests."""
    frame = frame.copy(deep=True)
    before_count = len(frame)
    frame["join_game_name_key"] = frame["join_game_name_key"].map(normalize_game_reference)
    frame = frame.loc[frame["join_game_name_key"].notna()].copy()
    invalid_key_count = before_count - len(frame)
    if frame.empty:
        raise ValueError("No valid canonical game keys remain after preprocessing.")
    flags = ["has_sales_match", "has_rental_match", "has_eilers_ts_match"]
    for name in flags:
        frame[name] = frame[name].map(as_bool)
    frame["parent_own_status"] = frame["parent_own_status"].map(normalize_game_reference).fillna("__unknown__")
    frame["_document_tiebreak"] = [
        hashlib.sha256(json.dumps(record, sort_keys=True).encode()).hexdigest()
        for record in json.loads(frame.to_json(orient="records", date_format="iso"))
    ]
    frame = frame.sort_values(["game_name", "_document_tiebreak"], kind="stable", na_position="last")
    mappings = list(zip(flags, ["to_sales_name_match", "to_rental_name_match", "to_eilers_ts_name_match"]))
    merged = []
    conflicting_mappings = 0
    for _, group in frame.groupby("join_game_name_key", sort=True):
        row = group.iloc[0].drop(labels=["_document_tiebreak"]).copy()
        parent_names = group["parent_game"].map(normalize_game_reference).dropna()
        row["parent_game"] = None if parent_names.empty else parent_names.iloc[0]
        for flag, name in mappings:
            row[flag] = bool(group[flag].any())
            usable = group.loc[group[flag], name].map(normalize_game_reference).notna()
            names = group.loc[group[flag], name].loc[usable]
            row[name] = None if names.empty else str(names.iloc[0]).strip()
            conflicting_mappings += int(names.map(normalize_game_reference).nunique() > 1)
        merged.append(row)
    frame = pd.DataFrame(merged).reset_index(drop=True)
    numeric_columns = list(dict.fromkeys(c for fields in feature_groups.values() for c in fields))
    release_age = pd.to_numeric(frame["max_months_since_release"], errors="coerce").replace([np.inf, -np.inf], np.nan)
    missing_numeric_counts = {}
    for column in numeric_columns:
        numeric_values = pd.to_numeric(frame[column], errors="coerce").replace([np.inf, -np.inf], np.nan)
        missing_numeric_counts[column] = int(numeric_values.isna().sum())
        frame[column] = numeric_values.fillna(0.0)
    frame["max_months_since_release"] = release_age
    frame["game_release_date"] = pd.to_datetime(frame["game_release_date"], errors="coerce")
    records = frame.to_dict("records")
    names = frame["join_game_name_key"].astype(str).to_numpy()
    flags_by_source = {source: frame[flag].to_numpy(dtype=bool) for source, (flag, _) in zip(["sales", "rental", "eilers"], mappings)}
    mapping_columns = {"sales": "to_sales_name_match", "rental": "to_rental_name_match", "eilers": "to_eilers_ts_name_match"}
    valid_names = {source: frame[column].map(normalize_game_reference).notna().to_numpy(dtype=bool)
                   for source, column in mapping_columns.items()}
    parent_keys = frame["parent_game"].map(normalize_game_reference).to_numpy()
    canonical_lookup = {name: index for index, name in enumerate(names)}
    aliases = {}
    for index, record in enumerate(records):
        for column in mapping_columns.values():
            alias = normalize_game_reference(record[column])
            if alias is not None:
                aliases.setdefault(alias, set()).add(index)
    resolutions = []
    for target_index, parent_key in enumerate(parent_keys):
        if parent_key is None:
            resolutions.append((None, "no_parent"))
            continue
        matches = {canonical_lookup[parent_key]} if parent_key in canonical_lookup else aliases.get(parent_key, set())
        if len(matches) != 1:
            resolutions.append((None, "ambiguous_name" if matches else "not_found"))
            continue
        parent_index = next(iter(matches))
        resolutions.append((None, "self_reference") if parent_index == target_index else (int(parent_index), "resolved"))
    return {
        "frame": frame, "records": records, "game_names": names,
        "feature_groups": copy.deepcopy(feature_groups), "numeric_columns": numeric_columns,
        "release_dates": frame["game_release_date"].to_numpy(dtype="datetime64[ns]"),
        "flags": flags_by_source, "valid_names": valid_names, "mapping_columns": mapping_columns,
        "parent_keys": parent_keys, "has_parent": np.array([key is not None for key in parent_keys], dtype=bool),
        "parent_resolutions": resolutions,
        "preprocessing": {"input_rows": before_count, "invalid_key_rows": invalid_key_count,
                          "canonical_games": len(frame), "duplicates_merged": before_count - invalid_key_count - len(frame),
                          "conflicting_mapping_groups": conflicting_mappings,
                          "missing_or_nonfinite_numeric_by_feature": missing_numeric_counts,
                          "missing_or_nonfinite_numeric_values": sum(missing_numeric_counts.values()),
                          "deduplication": "canonical key; alphabetical document then content hash; merge source flags/mappings",
                          "numeric_missing_value": 0.0},
    }


def weighted_distances(target_vector, candidate_vectors, method):
    """Full-vector distance with deterministic zero-vector cosine behavior."""
    if method == "euclidean":
        return np.linalg.norm(candidate_vectors - target_vector, axis=1)
    if method != "cosine":
        raise ValueError("Distance method must be cosine or euclidean.")
    target_norm = np.linalg.norm(target_vector)
    norms = np.linalg.norm(candidate_vectors, axis=1)
    similarities = np.zeros(len(candidate_vectors), dtype=np.float64)
    np.divide(np.einsum("ij,j->i", candidate_vectors, target_vector), target_norm * norms,
              out=similarities, where=target_norm * norms > 0)
    similarities[(target_norm == 0) & (norms == 0)] = 1.0
    return np.clip(1.0 - np.clip(similarities, -1.0, 1.0), 0.0, 1.0)


def select_neighbors_reference(candidates, target_mapping, cutoff, minimum):
    """Pure reference implementation of source deduplication and cutoff/fallback."""
    def key(value):
        if value is None:
            return None
        name = " ".join(str(value).lower().strip().split())
        return None if name in {"", "none", "null", "nan", "na", "n/a", "<na>"} else name

    best = {}
    for candidate in candidates:
        mapping = key(candidate.get("matched_source_name"))
        distance = candidate.get("distance")
        if mapping is None or mapping == key(target_mapping) or distance is None:
            continue
        distance = float(distance)
        if not math.isfinite(distance) or distance < 0:
            continue
        item = dict(candidate, distance=distance, _mapping=mapping)
        order = (distance, str(item.get("neighbor_game", item.get("join_game_name_key", ""))), str(item["matched_source_name"]))
        if mapping not in best or order < best[mapping][0]:
            best[mapping] = (order, item)
    ranked = sorted((item for _, item in best.values()), key=lambda item: (
        item["distance"], str(item.get("neighbor_game", item.get("join_game_name_key", ""))), item["_mapping"]))
    return [dict(item, neighbor_rank=index + 1, retained_within_cutoff=item["distance"] <= cutoff,
                 retained_as_min_neighbor_fallback=item["distance"] > cutoff)
            for index, item in enumerate(ranked) if item["distance"] <= cutoff or index < minimum]


def _functions():
    from pyspark.sql import functions as F
    return F


def _normalized(column):
    F = _functions()
    return F.regexp_replace(F.lower(F.trim(column.cast("string"))), r"\s+", " ")


def _valid_name(column):
    value = _normalized(column)
    return value.isNotNull() & ~value.isin("", "none", "null", "nan", "na", "n/a", "<na>")


def _finite(column):
    F = _functions()
    value = column.cast("double")
    return F.when(value.isNotNull() & ~F.isnan(value) & (F.abs(value) != float("inf")), value)


In [0]:
def monthly_eilers(raw, settings):
    from pyspark.sql import functions as F
    metrics = settings["performance_metrics"]
    pop = (_normalized(F.col("own_status"))).alias("own_status")
    frame = raw.select(_normalized(F.col("game_name")).alias("source_key"), pop,
                       F.trunc(F.to_date(F.col("yearmonth").cast("string"), "yyyy-MM-dd"), "month").alias("source_month"),
                       _finite(F.col("no_of_slots")).alias("slots"), *[_finite(F.col(m)).alias(m) for m in metrics])
    frame = frame.where(F.col("source_month").isNotNull() & (F.col("slots") > 0))
    if settings["slot_counts"]:
        frame = frame.where(F.col("slots").isin(settings["slot_counts"]))
    keys = ["source_key", "own_status", "source_month"]
    buckets = frame.groupBy(*keys, "slots").agg(*[F.avg(m).alias(m) for m in metrics])
    weighted = []
    for m in metrics:
        weights = F.when(F.col(m).isNotNull(), F.col("slots"))
        total = F.sum(weights)
        weighted.append(F.when(total > 0, F.sum(weights * F.col(m)) / total).alias(m))
    return buckets.groupBy(*keys).agg(*weighted)


def monthly_commercial(raw, channel, lease_column):
    from pyspark.sql import functions as F
    if channel == "sales":
        raw = raw.where(_normalized(F.col("revenue_type")).isin("sale", "sales"))
        key, date_col, quantity = "normalized_game_name", "beginning_month_date", "expr1_sum"
    else:
        key, date_col, quantity = "theme_name_eilers", "month", lease_column
    return (raw.select(_normalized(F.col(key)).alias("source_key"), F.trunc(F.to_date(date_col), "month").alias("source_month"),
                         _finite(F.col(quantity)).alias("quantity"))
              .where(_valid_name(F.col("source_key")) & F.col("source_month").isNotNull() & F.col("quantity").isNotNull())
              .groupBy("source_key", "source_month").agg(F.sum("quantity").alias("quantity")))


def annual_source_totals(raw, channel, lease_column):
    from pyspark.sql import functions as F
    months = monthly_commercial(raw, channel, lease_column)
    return (months.withColumn("year", F.year("source_month")).groupBy("year", "source_key")
            .agg(F.sum("quantity").alias("annual_total"), F.count("*").alias("observed_months"),
                 F.min("source_month").alias("first_month"), F.max("source_month").alias("last_month")).toPandas())


def load_debug_sources(spark, path):
    cfg = load_forecast_config(path)
    common = cfg["common"]
    ai_version = resolve_similarity_source_version(spark, common)
    versions = resolve_forecast_source_versions(spark, cfg)
    ai_path = common["data"]["ai_attributes_table"]
    ai = spark.read.option("versionAsOf", ai_version).table(ai_path)
    prepared = prepare_games(ai)
    raw = {key:spark.read.option("versionAsOf", versions[key]).table(table) for key, table in cfg["sources"].items()}
    print(f"{len(prepared['frame'])} canonical games. AI attributes v{ai_version}. "
          + ", ".join(f"{k} v{v}" for k, v in sorted(versions.items())) + ".")
    return cfg, prepared, raw


In [0]:
def choose_neighbors(prepared, game, settings, channel, forecast, mechanic=None, multiplier=1.0):
    """Run 05 for one target, then apply the 06 distance cutoff and fallback."""
    names = prepared["game_names"]
    i = int(np.flatnonzero(names == game)[0])
    row = prepared["records"][i]
    groups = prepared["feature_groups"]
    weights = {k: float(settings["raw_weights"].get(k, 0)) for k in groups}
    if any(not np.isfinite(v) or v < 0 for v in weights.values()) or sum(weights.values()) <= 0:
        raise ValueError("Enter finite, nonnegative weights with at least one positive group.")
    if not np.isfinite(multiplier) or multiplier < 0:
        raise ValueError("Enter a finite, nonnegative mechanic multiplier.")
    origin = pd.to_datetime(row["game_release_date"])
    if pd.isna(origin):
        origin = pd.to_datetime(settings.get("prediction_as_of_date"))
    audit_columns = ["List", "05 rank", "Game", "Matched name", "Distance", "Similarity", "06 rank", "Decision", "Parent"]
    empty = pd.DataFrame(columns=audit_columns)
    if pd.isna(origin):
        return empty, origin, "Missing prediction date"
    dates = prepared["release_dates"]
    past = dates < origin.to_datetime64()
    if not past.any():
        return empty, origin, "No earlier releases for historical scaling"

    fields, factors = [], []
    for group, columns in groups.items():
        if weights[group] == 0:
            continue
        within = np.ones(len(columns))
        if group == "mechanics" and mechanic in columns:
            within[columns.index(mechanic)] = multiplier
        if within.sum() == 0:
            raise ValueError("At least one mechanic must have a positive multiplier when its group has weight.")
        fields.extend(columns)
        factors.extend(np.sqrt(weights[group] / sum(weights.values()) * within / within.sum()))
    X = prepared["frame"][fields].fillna(0).to_numpy(dtype=float)
    low, high = X[past].min(axis=0), X[past].max(axis=0)
    scaled = np.zeros_like(X)
    np.divide(X - low, high - low, out=scaled, where=high != low)
    scaled = np.clip(scaled, 0, 1) * np.asarray(factors)
    distance = weighted_distances(scaled[i], scaled, settings["distance_method"])
    flags, valid = prepared["flags"], prepared["valid_names"]
    cutoff_date = pd.Timestamp(settings["neighbor_released_after"]).to_datetime64()
    latest = (origin - pd.DateOffset(months=settings["minimum_age_gap_months"])).to_datetime64()
    pool = past & (dates > cutoff_date) & (dates <= latest) & flags["eilers"] & valid["eilers"]
    pool[i] = False
    process = settings["parent_game_process"]
    if process == "p1_parent_games_only" and prepared["has_parent"][i]:
        pool &= prepared["has_parent"]
    parent, _ = prepared["parent_resolutions"][i]
    output = []
    for source in ("sales" if channel == "sales" else "rental", "eilers"):
        indices = np.flatnonzero(pool & flags[source] & valid[source])
        ranked = list(indices[np.lexsort((names[indices], distance[indices]))])
        if (process == "p3_include_parent" and source != "eilers" and parent is not None
                and past[parent] and flags[source][parent] and valid[source][parent]):
            ranked = [parent] + [j for j in ranked if j != parent]
        mapping = prepared["mapping_columns"][source]
        target_mapping = normalize_game_reference(row[mapping])
        seen = {target_mapping}
        candidates = []
        for j in ranked:
            if len(candidates) >= settings["neighbor_limits"][source]:
                break
            matched = normalize_game_reference(prepared["records"][j][mapping])
            if matched is None or matched in seen:
                continue
            seen.add(matched)
            candidates.append(dict(neighbor_game=names[j], matched_source_name=matched, distance=float(distance[j])))
        target_06 = (target_mapping or game) if source == "eilers" else target_mapping
        kept = select_neighbors_reference(candidates, target_06, forecast["distance_cutoff"], forecast["minimum_neighbors"])
        retained = {c["matched_source_name"]: c for c in kept}
        for rank, c in enumerate(candidates, 1):
            chosen = retained.get(c["matched_source_name"])
            reason = ("Within cutoff" if chosen["retained_within_cutoff"] else "Minimum-neighbor fallback") if chosen else "Dropped in 06"
            output.append([source, rank, c["neighbor_game"], c["matched_source_name"], c["distance"],
                           max(0., 1. - c["distance"]), chosen["neighbor_rank"] if chosen else None,
                           reason, parent is not None and c["neighbor_game"] == names[parent]])
    return pd.DataFrame(output, columns=audit_columns), origin, "Ready" if output else "No eligible neighbors"


def game_performance(eilers_monthly, prepared, game, channel, origin, audit, metrics, horizon):
    """01 = target prediction month; neighbor 01 = its first as-of source month."""
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window

    games = prepared["frame"].set_index("join_game_name_key")
    retained = audit.loc[audit["06 rank"].notna(), "Game"].drop_duplicates().tolist()
    roster = pd.DataFrame({"Game": [game, *retained]})
    roster["Role"] = ["Target"] + ["Neighbor"] * len(retained)
    roster["Eilers name"] = roster["Game"].map(games["to_eilers_ts_name_match"]).map(normalize_game_reference)
    columns = ["Game", "Role", "Source month", "Relative month", *metrics]
    frames = []
    if pd.isna(origin):
        return roster, pd.DataFrame(columns=columns)
    start = pd.Timestamp(origin).to_period("M").start_time
    pop = "owned" if channel == "sales" else "leased"
    base = eilers_monthly.where(F.col("own_status") == pop)
    for role, group in roster.groupby("Role", sort=False):
        keys = group["Eilers name"].dropna().unique().tolist()
        if not keys:
            continue
        f = base.where(F.col("source_key").isin(keys))
        if role == "Target":
            f = f.where((F.col("source_month") >= F.lit(start.date())) &
                        (F.col("source_month") < F.lit((start + pd.DateOffset(months=horizon)).date())))
            first = F.lit(start.date())
        else:
            f = f.where(F.col("source_month") < F.lit(start.date()))
            first = F.min("source_month").over(Window.partitionBy("source_key"))
        f = f.withColumn("month_index", F.months_between("source_month", first).cast("int") + 1)
        monthly = (f.where(F.col("month_index").between(1, horizon))
                   .select("source_key", "source_month", "month_index", *metrics).toPandas())
        if not monthly.empty:
            joined = group.merge(monthly, left_on="Eilers name", right_on="source_key")
            frames.append(joined.rename(columns={"source_month":"Source month", "month_index":"Relative month"})[columns])
    return roster, pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)


In [0]:
BRACKETS = ["Bottom 20%", "Middle 60%", "Top 20%"]
PRESENCE_GROUPS = {"themes": "theme", "mechanics": "mechanic"}


def game_families(prepared):
    """Keep connected parent/child titles and siblings in one fold or bootstrap cluster."""
    names = prepared["game_names"]
    links = {}
    def root(key):
        while key in links:
            key = links[key]
        return key
    for i, name in enumerate(names):
        parent, _ = prepared["parent_resolutions"][i]
        ref = prepared["parent_keys"][i]
        if ref is not None:
            a = root(name)
            b = root(names[parent] if parent is not None else ("unresolved parent", ref))
            if a != b:
                links[a] = b
    return {name: str(root(name)) for name in names}


def annual_cohort(prepared, totals, year, channel, minimum):
    frame = prepared["frame"].copy()
    mapping = "to_sales_name_match" if channel == "sales" else "to_rental_name_match"
    frame["source_key"] = frame[mapping].map(normalize_game_reference)
    mapped = frame.loc[frame["source_key"].notna()].copy()
    duplicates = mapped["source_key"].duplicated(keep=False)
    observations = totals.loc[totals["year"].eq(year)].copy()
    eligible = mapped.loc[~duplicates].merge(observations, on="source_key", how="inner", validate="one_to_one")
    eligible = eligible.loc[eligible["observed_months"].ge(minimum)].copy()
    ranks = eligible["annual_total"].rank(method="average")
    percentile = (ranks-1) / (len(ranks)-1) if len(ranks) > 1 else pd.Series(.5, index=ranks.index)
    eligible["bracket"] = np.select([percentile.le(.2), percentile.ge(.8)], [BRACKETS[0], BRACKETS[2]], default=BRACKETS[1])
    eligible["family"] = eligible["join_game_name_key"].map(game_families(prepared))
    matched = observations["source_key"].isin(mapped["source_key"])
    ambiguous = observations["source_key"].isin(mapped.loc[duplicates, "source_key"])
    counts = dict(source_names=len(observations), unmatched=int((~matched).sum()), ambiguous=int(ambiguous.sum()),
                  insufficient=int((matched & ~ambiguous & observations["observed_months"].lt(minimum)).sum()))
    return eligible.sort_values("join_game_name_key").reset_index(drop=True), counts


def bracket_range(X, outcome, families, draws=600):
    """Pointwise family-bootstrap intervals for the observed top-minus-bottom gap."""
    _, cluster = np.unique(families, return_inverse=True)
    g = cluster.max()+1
    empty = np.full(X.shape[1], np.nan)
    if min(len(set(cluster[outcome == j])) for j in (0, 2)) < 2:
        return empty, empty
    sums, counts = [], []
    for j in (0, 2):
        mask = outcome == j
        blocks = np.zeros((g, X.shape[1]))
        np.add.at(blocks, cluster[mask], X[mask])
        sums.append(blocks)
        counts.append(np.bincount(cluster[mask], minlength=g))
    rng = np.random.default_rng(324)
    samples = rng.multinomial(g, np.full(g, 1/g), size=draws)
    denom0, denom2 = (samples @ n for n in counts)
    valid = (denom0 > 0) & (denom2 > 0)
    if valid.sum() < draws // 2:
        return empty, empty
    diffs = ((samples[valid] @ sums[1]) / denom2[valid, None]
             - (samples[valid] @ sums[0]) / denom0[valid, None])
    return tuple(np.quantile(diffs, [.025, .975], axis=0))


def heldout_importance(X, y, families, controls):
    """Permute one feature at a time on held-out families. Errors are in bracket steps."""
    from sklearn.linear_model import Ridge
    from sklearn.model_selection import GroupKFold
    from sklearn.preprocessing import StandardScaler

    n, p = X.shape
    delta, positive = np.zeros(p), np.zeros(p, dtype=int)
    if n < 25 or len(set(families)) < 5 or np.bincount(y, minlength=3).min() < 4:
        return np.full(p, np.nan), np.full(p, np.nan), None
    predictors = np.column_stack([X, controls])
    loss = baseline_loss = 0.
    rng = np.random.default_rng(103)
    for train, test in GroupKFold(n_splits=5).split(predictors, y, families):
        scale = StandardScaler().fit(predictors[train])
        model = Ridge(alpha=10., solver="cholesky").fit(scale.transform(predictors[train]), y[train])
        xt = scale.transform(predictors[test])
        fitted = model.predict(xt)
        residual = fitted - y[test]
        original = np.abs(residual)
        fold_delta = np.zeros(p)
        for _ in range(20):
            # A common row shuffle allows direct comparisons across feature columns.
            change = (xt[rng.permutation(len(test)), :p] - xt[:, :p]) * model.coef_[:p]
            fold_delta += (np.abs(residual[:, None] + change) - original[:, None]).sum(axis=0)
        fold_delta /= 20
        positive += fold_delta > 1e-9
        delta += fold_delta
        loss += original.sum()
        baseline_loss += np.abs(y[test] - np.median(y[train])).sum()
    return delta / n, positive, (loss/n, baseline_loss/n)


def feature_report(cohort, groups, year):
    group_for = {col:g for g, cols in groups.items() if g != "months_since_release" for col in cols}
    columns = list(group_for)
    data = cohort[columns].astype(float).fillna(0).copy()
    binary = np.array([group_for[col] in PRESENCE_GROUPS for col in columns], dtype=bool)
    data.loc[:, binary] = data.loc[:, binary].gt(0).astype(float)
    X = data.to_numpy()
    y = cohort["bracket"].map(dict(zip(BRACKETS, [0,1,2]))).to_numpy(dtype=int)
    families = cohort["family"].to_numpy()
    table = pd.DataFrame({"Feature":columns, "Group":[group_for[c] for c in columns]})
    unit = np.where(binary, 100., 1.)
    table["Units"] = np.where(binary, "% present", "mean value")
    for j, label in enumerate(BRACKETS):
        table[label] = X[y == j].mean(axis=0) * unit if (y == j).any() else np.nan
    gap = table[BRACKETS[2]] - table[BRACKETS[0]]
    table["Top − bottom"] = gap
    spread = X.std(axis=0)
    table["Gap in SDs"] = np.divide(gap / unit, spread, out=np.full(len(columns),np.nan), where=spread>0)
    lo, hi = bracket_range(X, y, families)
    table["Gap low 95%"], table["Gap high 95%"] = lo*unit, hi*unit
    table["Games above zero"] = (X > 0).sum(axis=0)
    table["Games at zero"] = (X == 0).sum(axis=0)
    release = pd.to_datetime(cohort["game_release_date"])
    # Use age at the selected year, never today's snapshot age, as an exposure control.
    age = (year - release.dt.year) * 12 + 1 - release.dt.month
    controls = np.column_stack([cohort["observed_months"], age.fillna(0), age.isna().astype(float)])
    impact, folds, errors = heldout_importance(X, y, families, controls)
    table["Held-out MAE increase"] = impact
    table["Positive folds (of 5)"] = pd.array(folds, dtype="Int64")
    # Each theme or mechanic needs enough games in both presence states.
    scarce = binary & ((X > 0).sum(axis=0) < 5)
    scarce |= binary & ((X == 0).sum(axis=0) < 5)
    consistent = np.zeros(len(columns),dtype=bool)
    if errors is not None and errors[0] < errors[1]:
        consistent = ((impact >= .02) & (folds >= 4) & ((lo > 0) | (hi < 0)))
    sparse_label = "Sparse " + table["Group"].map(PRESENCE_GROUPS).fillna("")
    table["Evidence"] = np.select(
        [spread == 0, scarce, consistent, np.isnan(impact)],
        ["No variation", sparse_label, "Consistent in this cohort", "Descriptive only"],
        default="Mixed or weak")
    return table, errors


In [0]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import HTML, display, clear_output
from html import escape


def show_data(frame, details=None):
    content = '<div style="overflow:auto;max-height:440px">' + frame.to_html(
        index=False, border=0, escape=True, na_rep="—", float_format=lambda x:f"{x:,.4g}") + '</div>'
    if details:
        content = "<details><summary>" + escape(details) + "</summary>" + content + "</details>"
    display(HTML(content))


def choice(title, options, **kwargs):
    return widgets.Dropdown(description=title, options=options, style={"description_width":"initial"},
                            layout=widgets.Layout(width="min(100%, 680px)"), **kwargs)


def commercial_curve(monthly, source_key, start, horizon, channel, is_target):
    """Align one commercial history to the chart's relative calendar months."""
    values = monthly.loc[monthly["source_key"].eq(source_key)].set_index("source_month")["quantity"].sort_index()
    if is_target:
        values = values.loc[values.index >= start]
        first = start
    else:
        values = values.loc[values.index < start]
        first = values.index.min()
    if values.empty:
        return pd.Series(dtype=float)
    last = min(values.index.max(), first + horizon - 1)
    months = pd.period_range(first, last, freq="M")
    observed = values.reindex(months)
    curve = observed.fillna(0).cumsum() if channel == "sales" else observed.ffill()
    if is_target:
        curve = curve.where(observed.notna())
    curve.index = pd.RangeIndex(1, len(curve) + 1, name="Relative month")
    return curve


def draw_demand(raw, prepared, game, channel, origin, audit, settings):
    from pyspark.sql import functions as F

    sales = channel == "sales"
    source = "sales" if sales else "rental"
    label = "Cumulative sales (units)" if sales else settings["lease_curve_label"]
    if pd.isna(origin):
        print(f"{label} is unavailable because {game} has no prediction date.")
        return
    retained = audit.loc[audit["List"].eq(source) & audit["06 rank"].notna()].sort_values("06 rank")
    mapping = prepared["mapping_columns"][source]
    target_key = normalize_game_reference(prepared["frame"].set_index("join_game_name_key").at[game, mapping])
    keys = retained["Matched name"].tolist()
    if target_key is not None:
        keys.append(target_key)
    if not keys:
        print(f"{game} has no {source} mapping and no retained {source} neighbors.")
        return
    start = pd.Period(origin, freq="M")
    horizon = settings["horizon_months"]
    monthly = (monthly_commercial(raw[channel], channel, settings["lease_value_column"])
               .where(F.col("source_key").isin(keys))
               .where(F.col("source_month") < F.lit((start + horizon).start_time.date()))
               .toPandas())
    monthly["source_month"] = pd.to_datetime(monthly["source_month"]).dt.to_period("M")
    target = commercial_curve(monthly, target_key, start, horizon, channel, is_target=True)
    neighbors = pd.DataFrame({
        name: commercial_curve(monthly, key, start, horizon, channel, is_target=False)
        for name, key in retained[["Game", "Matched name"]].itertuples(index=False, name=None)
    }).dropna(axis=1, how="all")

    if not target.notna().any():
        reason = "no source-name mapping" if target_key is None else "no observed months in the forecast window"
        print(f"No {source} actuals for {game} ({reason}).")
    if neighbors.empty:
        print(f"No retained {source} neighbors with history before {start}.")
        if not target.notna().any():
            return

    fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)
    if not neighbors.empty:
        for name, values in neighbors.items():
            ax.plot(values.index, values, linewidth=1.2, alpha=.75, marker=".", label=name)
        baseline = neighbors.mean(axis=1)
        deviation = neighbors.std(axis=1, ddof=0)
        ax.fill_between(baseline.index, (baseline - 2 * deviation).clip(lower=0).to_numpy(),
                        (baseline + 2 * deviation).to_numpy(), color="#6385aa", alpha=.13, label="Neighbor mean ±2 SD")
        ax.plot(baseline.index, baseline, color="#245584", linestyle="--", marker="o",
                markersize=4, linewidth=2.3, label="Forecast · neighbor mean")
    if target.notna().any():
        ax.plot(target.index, target, color="#161616", marker="D", linewidth=2.8, label=f"Actual · {game}")
    ax.set(title=f"{label} · {game}", ylabel=label, xlabel="Relative month", xticks=range(1, horizon + 1))
    ax.grid(alpha=.17)
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1), fontsize=8, frameon=False)
    display(fig)
    plt.close(fig)


def draw_performance(roster, observations, metrics, horizon):
    roster = roster.copy()
    observed = observations.dropna(subset=metrics, how="all")
    counts = observed.groupby("Game")["Relative month"].nunique()
    roster["Observed Eilers months"] = roster["Game"].map(counts).fillna(0).astype(int)
    show_data(roster)
    if observations.empty:
        print("No Eilers observations for the selected games, ownership and time windows.")
        return
    missing = roster.loc[~roster["Game"].isin(observed["Game"]), "Game"]
    if not missing.empty:
        print("No observed index values for: " + ", ".join(missing))
    months = range(1, horizon+1)
    colors = {n:plt.get_cmap("tab20")(i % 20) for i,n in enumerate(roster.loc[roster.Role.eq("Neighbor"),"Game"])}
    fig, axes = plt.subplots(len(metrics), 1, figsize=(12, 3.5*len(metrics)), squeeze=False, constrained_layout=True)
    for metric, ax in zip(metrics, axes[:, 0]):
        for (role, name), series in observations.groupby(["Role", "Game"]):
            values = series.set_index("Relative month")[metric].reindex(months)
            if values.notna().any():
                target = role == "Target"
                ax.plot(months, values, color="#161616" if target else colors[name],
                        linewidth=2.8 if target else 1.3, marker="D" if target else ".",
                        zorder=5 if target else 2, label=f"Target · {name}" if target else name)
        ax.set(title=metric.replace("avg_", "").replace("_", " "), xlabel="Relative month", ylabel="Index", xticks=list(months))
        ax.grid(alpha=.17)
        if ax.lines:
            ax.legend(loc="upper left", bbox_to_anchor=(1,1), fontsize=8, frameon=False)
    display(fig)
    plt.close(fig)
    show_data(observations.sort_values(["Role","Game","Relative month"]), "Observed monthly values and actual calendar dates")


def launch_neighbors(cfg, prepared, raw):
    presets = {e["name"]:e["settings"] for e in cfg["common"]["experiments"]}
    initial = next((e["name"] for e in cfg["common"]["experiments"] if e["enabled"]), next(iter(presets)))
    preset = choice("Starting experiment", list(presets), value=initial)
    frame = prepared["frame"]
    names = sorted(prepared["game_names"])
    dated = frame.loc[frame["has_eilers_ts_match"] & frame["game_release_date"].notna()].sort_values("game_release_date")
    default = dated.iloc[-1]["join_game_name_key"] if not dated.empty else names[0]
    target = widgets.Combobox(description="Target game", options=names, value=default, ensure_option=True,
                              layout=widgets.Layout(width="min(100%, 680px)"))
    channel = choice("Population", [("Sale / owned","sales"),("Rent / leased","lease")])
    parent = choice("Parent rule", sorted(PARENT_PROCESSES))
    method = choice("Distance", ["cosine","euclidean"])
    weights = {g:widgets.FloatText(description=g.replace("_"," "), style={"description_width":"initial"},
                                   layout=widgets.Layout(width="210px")) for g in prepared["feature_groups"]}
    mechanics = prepared["feature_groups"]["mechanics"]
    mechanic = choice("Individual mechanic", [("All at standard weight", "")]+[(m,m) for m in mechanics])
    multiplier = widgets.FloatText(description="Multiplier", value=1., layout=widgets.Layout(width="220px"))
    run = widgets.Button(description="Find like games", button_style="primary")
    out = widgets.Output()
    eilers = monthly_eilers(raw["performance"],cfg["settings"])
    def reset(_=None):
        settings = presets[preset.value]
        for g, box in weights.items():
            box.value = settings["raw_weights"].get(g,0.)
        parent.value, method.value = settings["parent_game_process"], settings["distance_method"]
        mechanic.value, multiplier.value = "", 1.
    def render(_=None):
        with out:
            clear_output(wait=True)
            try:
                settings = dict(presets[preset.value], raw_weights={g:b.value for g,b in weights.items()},
                                parent_game_process=parent.value, distance_method=method.value,
                                prediction_as_of_date=cfg["common"]["data"]["prediction_as_of_date"])
                game = normalize_game_reference(target.value)
                if game not in names:
                    print("Select a canonical game from the list.")
                    return
                audit, origin, status = choose_neighbors(prepared, game, settings, channel.value, cfg["settings"],
                                                        mechanic.value or None, multiplier.value)
                settings06 = cfg["settings"]
                date_text = "missing" if pd.isna(origin) else str(pd.Timestamp(origin).date())
                print(f"{game} | {channel.label} | {settings['parent_game_process']} | {settings['distance_method']} | prediction date {date_text} | {status}")
                effective = pd.Series(settings["raw_weights"],dtype=float)
                show_data(pd.DataFrame({"Feature group":effective.index,"Entered weight":effective.values,
                                        "Group share (%)":100*effective.values/effective.sum()}))
                if mechanic.value:
                    print(f"{mechanic.value} × {multiplier.value:g} within the mechanics group.")
                print(f"05 release cutoff {settings['neighbor_released_after']}, gap {settings['minimum_age_gap_months']} months. "
                      f"06 distance cutoff {settings06['distance_cutoff']}, minimum {settings06['minimum_neighbors']} when available.")
                if not audit.empty:
                    show_data(audit)
                draw_demand(raw, prepared, game, channel.value, origin, audit, settings06)
                roster, obs = game_performance(eilers, prepared, game, channel.value, origin, audit,
                                               settings06["performance_metrics"], settings06["horizon_months"])
                print("The target starts in its prediction month. Each retained neighbor starts in its first historical "
                      "Eilers month and uses only observations before the target's prediction month. Gaps are unobserved.")
                draw_performance(roster, obs, settings06["performance_metrics"], settings06["horizon_months"])
            except (ValueError, KeyError) as exc:
                print(str(exc))
    preset.observe(reset, names="value")
    run.on_click(render)
    reset()
    weight_boxes = widgets.Box(list(weights.values()), layout=widgets.Layout(display="flex",flex_flow="row wrap"))
    display(widgets.VBox([preset,target,channel,parent,method,weight_boxes,mechanic,multiplier,run,out]))
    render()
    return {"game":target, "weights":weights, "run":run, "output":out}


def launch_features(cfg, prepared, raw):
    from functools import lru_cache
    population = choice("Outcome", [("Sales · units in calendar year","sales"),
                                   ("Rent · summed monthly rental quantity","lease")])
    year = choice("Calendar year", [])
    minimum = widgets.IntSlider(description="Min. observed months", value=1, min=1, max=12,
                                style={"description_width":"initial"})
    groups = [g for g in prepared["feature_groups"] if g!="months_since_release"]
    feature_group = choice("Show features", [("All features","all")]+[(g.title(),g) for g in groups],value="mechanics")
    picked = choice("Inspect feature", [])
    run = widgets.Button(description="Analyze year",button_style="primary")
    out, detail = widgets.Output(), widgets.Output()
    state = {"report":None, "loading":False}
    @lru_cache(maxsize=2)
    def totals(channel):
        return annual_source_totals(raw[channel], channel, cfg["settings"]["lease_value_column"])
    def update_year(_=None):
        year.options = sorted(totals(population.value)["year"].dropna().astype(int).unique().tolist(), reverse=True)
    def show_detail(_=None):
        if state["loading"]:
            return
        with detail:
            clear_output(wait=True)
            if picked.value is None or state["report"] is None:
                print("Select a feature from the analyzed cohort.")
                return
            cohort, report = state["report"]
            row = report.loc[report.Feature.eq(picked.value)].iloc[0]
            x = np.arange(3)
            fig, ax = plt.subplots(figsize=(6.6, 3.1),constrained_layout=True)
            ax.bar(x, row[BRACKETS].astype(float), color=["#668697","#a3b6ac","#277766"])
            ax.set(xticks=x,xticklabels=BRACKETS,ylabel=row["Units"],title=row["Feature"])
            if row["Units"] == "% present":
                ax.set_ylim(0,100)
            display(fig)
            plt.close(fig)
            if row["Group"] in PRESENCE_GROUPS:
                kind = PRESENCE_GROUPS[row["Group"]]
                present = cohort[picked.value].fillna(0).gt(0)
                support = pd.crosstab(cohort["bracket"], present).reindex(index=BRACKETS,columns=[True,False],fill_value=0)
                support.columns = [f"Games with {kind}", f"Games without {kind}"]
                show_data(support.rename_axis("Bracket").reset_index())
    def show_report(_=None):
        with out:
            clear_output(wait=True)
            state["loading"] = True
            try:
                state["report"] = None
                picked.options = []
                if year.value is None:
                    print("No annual observations in this source.")
                    return
                channel, selected_year = population.value, int(year.value)
                source = totals(channel)
                annual = source.loc[source.year.eq(selected_year)]
                cohort, counts = annual_cohort(prepared, source, selected_year, channel, minimum.value)
                definition = "sales units" if channel == "sales" else "monthly " + cfg["settings"]["lease_value_column"] + " summed over the year"
                print(f"{selected_year} | {definition} | at least {minimum.value} observed month(s). "
                      f"Source months {pd.Timestamp(annual.first_month.min()):%Y-%m} to {pd.Timestamp(annual.last_month.max()):%Y-%m}.")
                print(f"{len(cohort)} games included from {counts['source_names']} observed source names. "
                      f"Excluded: {counts['unmatched']} unmapped, {counts['ambiguous']} ambiguous, "
                      f"{counts['insufficient']} below the month minimum.")
                if cohort.empty:
                    print("No matched games meet this coverage setting.")
                    return
                show_data(cohort.groupby("bracket").agg(Games=("annual_total","size"), Lowest=("annual_total","min"),
                          Median=("annual_total","median"), Highest=("annual_total","max"),
                          Median_observed_months=("observed_months","median")).reindex(BRACKETS).rename_axis("Bracket").reset_index())
                show_data(cohort[["join_game_name_key","source_key","annual_total","observed_months","bracket"]]
                          .sort_values("annual_total",ascending=False), "Games, actual quantities and bracket assignments")
                report, errors = feature_report(cohort, prepared["feature_groups"], selected_year)
                if errors is None:
                    print("Descriptive results only. The held-out check needs ≥25 games, ≥5 families and ≥4 games per bracket.")
                else:
                    print(f"Held-out bracket MAE: all features and controls {errors[0]:.3f}, training-median baseline "
                          f"{errors[1]:.3f} bracket steps. Positive permutation importance means prediction got worse after shuffling.")
                subset = report if feature_group.value=="all" else report.loc[report.Group.eq(feature_group.value)]
                subset = subset.assign(_gap=subset["Gap in SDs"].abs()).sort_values(
                    ["Held-out MAE increase","_gap","Feature"],ascending=[False,False,True],na_position="last")
                show_data(subset.drop(columns="_gap"))
                state["report"] = cohort, report
                picked.options = subset.Feature.tolist()
            finally:
                state["loading"] = False
                show_detail()
    population.observe(update_year, names="value")
    picked.observe(show_detail, names="value")
    run.on_click(show_report)
    update_year()
    display(widgets.VBox([population,year,minimum,feature_group,run,out,picked,detail]))
    show_report()
    return {"year":year, "outcome":population, "feature":picked, "run":run, "output":out}


In [0]:
debug_config, debug_games, debug_sources = load_debug_sources(spark, CONFIG_PATH)


## Choose a game and change its neighbors

Choose a preset, game, ownership population and group weights. The optional individual multiplier changes a mechanic's share *within* the mechanics group. A multiplier of 1 reproduces normal group weighting. The other similarity rules and source limits come from the selected preset.

The table shows the notebook 05 candidate rank and notebook 06 retention decision for each commercial and Eilers candidate. The first chart shows cumulative units sold for **Sale / owned** or the configured monthly rental quantity for **Rent / leased**. The black diamonds show available target actuals from the prediction month onward. The dashed line is the mean of the retained commercial neighbors with history in each relative month. The shaded band shows their spread (±2 population standard deviations, with a lower bound of zero). Neighbor sales accumulate from their first sales month. Within each neighbor’s historical coverage, a missing sales month adds zero and a missing rental month carries forward the previous level. Each neighbor’s curve ends at its last available month before the target prediction month or at the configured horizon, whichever comes first.

The Eilers charts below compare the target's observed coin-in and theoretical net win indices against each retained game's observed indices. The target uses the prediction month as month 1. Neighbors use their first historical observation as month 1. Look at the calendar dates under the chart to see exactly what was compared.


In [0]:
neighbor_debug = launch_neighbors(debug_config, debug_games, debug_sources)


## Which features separate top, middle and bottom games?

Select a calendar year and Sale or Rent. Sales uses total observed units. Rent sums the configured monthly rental quantity across observed months, so a cabinet can contribute in more than one month. With the default rental source this measures rental records across months. A shorter reporting window or a midyear release can affect totals. Use the observed-month filter and coverage table to check exposure. Each mapped game counts once. Source names mapped to multiple games are excluded.

Brackets use average percentile rank within the selected cohort. Equal totals stay together, so an extreme bracket can be small or empty. Each theme and mechanic shows the percentage of games in each bracket where it is present. Any positive score counts as present. Zero and missing scores count as absent. For these features, **Top − bottom** is the difference in percentage points. Other numeric features show their mean value. Its pointwise 95% interval resamples whole game families 600 times, keeping the observed brackets fixed. These intervals are exploratory across many features.

**Held-out MAE increase** measures how much harder it is to predict bracket position after shuffling one feature on held-out games. Bottom, middle and top have scores 0, 1 and 2. Five folds keep known parent/child families together. A fixed ridge model uses the same presence flags for themes and mechanics, the other content features, observed months and age at the start of the selected year. Standardization uses training games only. Current snapshot age is excluded from annual feature importance. Each held-out permutation is repeated 20 times. The baseline predicts the training median bracket.

**Consistent in this cohort** requires a nonzero top–bottom gap interval, a held-out error increase of at least 0.02 bracket steps, a positive increase in at least four folds, and a model that beats the baseline. Each theme or mechanic needs at least five games where it is present and five where it is absent. This label describes predictive evidence for this year's mapped games. Correlated features can share a signal, and a year's association cannot establish causation. Recheck another year before changing production weights.


In [0]:
annual_debug = launch_features(debug_config, debug_games, debug_sources)


In [0]:
import pandas as pd

SALES_ONLY = False

games = debug_games["frame"]
if SALES_ONLY:
    has_sales_name = games["to_sales_name_match"].map(normalize_game_reference).notna()
    games = games.loc[games["has_sales_match"] & has_sales_name]

if games.empty:
    raise ValueError("No games in the selected population.")

n_games = len(games)
summary_rows = []
feature_tables = []

for group in ["themes", "mechanics"]:
    cols = debug_games["feature_groups"][group]
    present = games[cols].fillna(0).gt(0)

    games_per_feature = present.sum(axis=0)
    features_per_game = present.sum(axis=1)

    summary_rows.append({
        "Group": group,
        "Games": n_games,
        "Features": len(cols),
        "Absent game-feature pairs (%)": 100 * (1 - present.to_numpy().mean()),
        "Median features present per game": features_per_game.median(),
        "Games with none present": features_per_game.eq(0).sum(),
        "Features never present": games_per_feature.eq(0).sum(),
        "Features present in 1–4 games": games_per_feature.between(1, 4).sum(),
    })

    feature_tables.append(pd.DataFrame({
        "Group": group,
        "Feature": cols,
        "Games present": games_per_feature.to_numpy(),
        "Games absent": n_games - games_per_feature.to_numpy(),
        "Present (%)": 100 * games_per_feature.to_numpy() / n_games,
    }))

sparsity_summary = pd.DataFrame(summary_rows).round(1)
feature_sparsity = (
    pd.concat(feature_tables, ignore_index=True)
    .sort_values(["Group", "Games present", "Feature"])
    .reset_index(drop=True)
)

print("Population:", "Sales-mapped games" if SALES_ONLY else "All canonical games")
display(sparsity_summary)
for group in ["themes", "mechanics"]:
    display(feature_sparsity.loc[feature_sparsity["Group"].eq(group)].round(1))